# **Урок 7**. Детекция с YOLO

В теоретической части мы с вами узнали о сложной и запутанной судьбе `YOLO` детекторов. 

Тем не менее, `YOLO` от Ultralytics - это самые удобные репозитории, который вы можете найти для задачи детекции. 

На данный момент два самых популярных репозитория — это <a href="https://github.com/ultralytics/yolov5">YOLOv5</a> (этот репозиторий развивается по сей день и&nbsp;даже имеет некоторые уникальные функции, которых нет в основном репозитории Ultralytics) и&nbsp;<a href="https://github.com/ultralytics/ultralytics">Ultralytics</a> (главный репозиторий, где собраны почти все модели YOLO начиная с версии 5).

Некоторые версии YOLO разрабатываются самими Ultralytics, например **YOLOv5**, **YOLOv8** и&nbsp;**YOLOv11** (самая последняя на данный момент). Однако многие модели созданы сторонними разработчиками, которые также вносят новаторские решения и&nbsp;обеспечивают высокие результаты. Ultralytics открыты к новым разработкам и&nbsp;часто добавляют в&nbsp;основной репозиторий модели от сторонних авторов, поддерживая так обширный хаб для&nbsp;всей семьи YOLO. Обычно процесс добавления внешней модели выглядит так: авторы новой версии YOLO делают форк текущего репозитория Ultralytics, проводят эксперименты, добавляют свои слои и&nbsp;подходы, а затем выкладывают модель под названием вроде YOLOvLAST+1. Если модель показывает действительно хорошие результаты, то со временем она добавляется в&nbsp;главный репозиторий Ultralytics и&nbsp;становится частью официальной документации.

Давайте в этой практике разберемся в устройстве проекта [Ultralytics](https://github.com/ultralytics/ultralytics) и научимся обучать сетки для собственных задач.

# **План**

1. [Форматы разметки для детекции](#section1).
2. [Знакомство с проектом Ultralytics и&nbsp;его структурой](#section2).
3. [Разбираемся с конфигурацией моделей](#section3).
4. [Разбираемся с конфигурацией обучения](#section4).
5. [Полезные механизмы для&nbsp;обучения](#section5).
    - 5.1 [Механизмы scale и&nbsp;multi_scale](#section5-1).
    - 5.2 [Механизмы mosaic и&nbsp;close_mosaic](#section5-2).
    - 5.3 [Распределение весов у лоссов во время обучения](#section5-3).
    - 5.4 [Критерий качества модели](#section5-4).
6. [Заводим обучение YOLO](#section6).
7. [Знакомимся с ONNX форматом](#section7).

In [ ]:
# Устанавливаем репозиторий к себе
%pip install ultralytics

In [ ]:
import math
import random

import cv2
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.colors as mcolors
import matplotlib.patches as patches
import matplotlib.pyplot as plt

from PIL import Image
from IPython.display import SVG, display
from torchvision.transforms import Resize, ToTensor

%matplotlib inline

<a id='section2'></a>
## **1. Форматы разметки для детекции**

При работе с задачами детекции мы сталкиваемся с разными форматами разметки данных. Основные форматы разметки:
   1. [Pascal VOC](http://host.robots.ox.ac.uk/pascal/VOC/)-формат использует абсолютные координаты в&nbsp;пикселях в&nbsp;формате `[xmin, ymin, xmax, ymax]`. Преимущества формата в&nbsp;его интуитивной понятности и&nbsp;лёгкой визуализации. Основные недостатки: зависимость от размера изображения и&nbsp;избыточность при хранении.

   2. [COCO](https://cocodataset.org/#format-data)-формат также использует абсолютные координаты в&nbsp;пикселях в&nbsp;формате `[x, y, width, height]`, где x, y — координаты левого верхнего угла. Является стандартом де-факто для&nbsp;многих датасетов, удобен для&nbsp;вычисления метрик и&nbsp;широко используется в&nbsp;компьютерном зрении.

   3. [YOLO](https://arxiv.org/abs/1506.02640)-формат использует нормализованные координаты (от 0 до 1) в&nbsp;формате `[x_center, y_center, width, height]`, где все значения делятся на ширину или высоту изображения. Отличается независимостью от размера изображения и удобством для обучения нейросетей

Почему `YOLO` использует свой формат:
1. Нормализация координат в `YOLO` приводит все значения к диапазону `[0; 1]`, что делает обучение более стабильным и&nbsp;существенно упрощает работу функции потерь за счёт единого масштаба значений.
2. Центрированные координаты лучше соответствуют архитектуре `YOLO`, где сетка разбивает изображение на ячейки, и&nbsp;каждая ячейка предсказывает объекты относительно своего центра, что улучшает точность детекции.
3. Преимущества при обучении включают единый масштаб всех значений, полную независимость от разрешения исходных изображений и&nbsp;возможность легко масштабировать изображения без необходимости пересчёта координат.


In [ ]:
def draw_stick_figure(ax, x, y, size=40, color='black'):
    """Нарисуем простого человечка)"""
    # Голова
    head = plt.Circle((x, y-size/2), size/8, color=color, fill=False)
    ax.add_artist(head)

    # Тело
    ax.plot([x, x], [y-size/2+size/8, y+size/8], color=color)

    # Ручки
    arm_y = y - size/4
    ax.plot([x-size/4, x+size/4], [arm_y, arm_y], color=color)

    # Ноги
    leg_top_y = y+size/8
    ax.plot([x, x+size/4], [leg_top_y, leg_top_y+size/3], color=color)
    ax.plot([x, x-size/4], [leg_top_y, leg_top_y+size/3], color=color)

    
def draw_road(ax, y_position, width, color='gray'):
    road = patches.Rectangle((0, y_position), width, 30, facecolor=color, alpha=0.3)
    ax.add_patch(road)

    dash_length = 20
    space_length = 15
    y_center = y_position + 15
    x_position = 0

    while x_position < width:
        ax.plot([x_position, x_position + dash_length],
                [y_center, y_center], color='white', linewidth=2)
        x_position += dash_length + space_length

        
def plot_bbox_formats(figsize=(15, 10)):
    img = np.ones((400, 600, 3)) * 240

    height, width = img.shape[:2]

    figure_width,figure_height = 70, 85
    center_x, center_y = 200, 150

    # VOC-формат (xmin, ymin, xmax, ymax)
    voc_bbox = [
        center_x - figure_width/2,   # xmin
        center_y - figure_height/2,  # ymin
        center_x + figure_width/2,   # xmax
        center_y + figure_height/2   # ymax
    ]

    # COCO-формат (x, y, width, height)
    coco_bbox = [
        center_x - figure_width/2,   # x
        center_y - figure_height/2,  # y
        figure_width,                # width
        figure_height                # height
    ]

    # YOLO-формат (x_center, y_center, width, height) — нормализованный
    yolo_bbox = [
        center_x/width,       # x_center
        center_y/height,      # y_center
        figure_width/width,   # width
        figure_height/height  # height
    ]

    _, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=figsize)

    # VOC-формат
    ax1.imshow(img/255 if img.max() > 1 else img)
    draw_road(ax1, 170, width)
    rect = patches.Rectangle((voc_bbox[0], voc_bbox[1]),
                           voc_bbox[2]-voc_bbox[0],
                           voc_bbox[3]-voc_bbox[1],
                           linewidth=2,
                           edgecolor='r',
                           facecolor='none')
    ax1.add_patch(rect)
    draw_stick_figure(ax1,
                     (voc_bbox[0] + voc_bbox[2])/2,
                     (voc_bbox[1] + voc_bbox[3])/2,
                     size=60)
    ax1.set_title('Pascal VOC формат\n(xmin, ymin, xmax, ymax)\n' +
                  f'{[int(x) for x in voc_bbox]}')

    # COCO формат
    ax2.imshow(img/255 if img.max() > 1 else img)
    draw_road(ax2, 170, width)
    rect = patches.Rectangle((coco_bbox[0], coco_bbox[1]),
                           coco_bbox[2],
                           coco_bbox[3],
                           linewidth=2,
                           edgecolor='g',
                           facecolor='none')
    ax2.add_patch(rect)
    draw_stick_figure(ax2,
                     coco_bbox[0] + coco_bbox[2]/2,
                     coco_bbox[1] + coco_bbox[3]/2,
                     size=60)
    ax2.set_title('COCO формат\n(x, y, width, height)\n' +
                  f'{[int(x) for x in coco_bbox]}')

    # YOLO-формат
    ax3.imshow(img/255 if img.max() > 1 else img)
    draw_road(ax3, 170, width)
    # Денормализуем YOLO-координаты обратно в пиксели для визуализации
    yolo_pixel = [
        (yolo_bbox[0] - yolo_bbox[2]/2) * width,  # x1
        (yolo_bbox[1] - yolo_bbox[3]/2) * height, # y1
        yolo_bbox[2] * width,  # width
        yolo_bbox[3] * height  # height
    ]
    rect = patches.Rectangle((yolo_pixel[0], yolo_pixel[1]),
                           yolo_pixel[2],
                           yolo_pixel[3],
                           linewidth=2,
                           edgecolor='b',
                           facecolor='none')
    ax3.add_patch(rect)
    draw_stick_figure(ax3,
                     yolo_bbox[0] * width,
                     yolo_bbox[1] * height,
                     size=60)
    ax3.set_title('YOLO формат\n(x_center/w, y_center/h, width/w, height/h)\n' +
                  f'[{", ".join([f"{x:.3f}" for x in yolo_bbox])}]')

    for ax in [ax1, ax2, ax3]:
        ax.axis('off')

    plt.tight_layout()
    plt.show()

plot_bbox_formats()

Итак, теперь мы знаем, что
- существуют разные форматы хранения разметки для задачи детекции
- основные из них: `Pascal VOC`, `COCO` и `YOLO`
- в `YOLO`, который нас и интересует, хранятся нормализованные координаты, что очень удобно при масштабировании изображений

<a id='section3'></a>
## **2. Знакомство с проектом Ultralytics и его структурой**

- **Архитектура репозитория [Ultralytics](https://github.com/ultralytics/ultralytics)**

  Главный репозиторий [Ultralytics](https://github.com/ultralytics/ultralytics) организован так, чтобы упростить настройку, обучение и&nbsp;развёртывание семейства моделей YOLO. Репозиторий построен на системе обратных вызовов (callbacks), что делает его модульным и&nbsp;гибким. Callbacks позволяют добавлять пользовательские функции на разных этапах — во&nbsp;время обучения, валидации или предсказания, — что упрощает настройку под задачи. Это помогает интегрировать дополнительные функции и&nbsp;расширять возможности без&nbsp;переписывания основного кода.

- **Директория конфигураций**

  Важная часть репозитория — директория [`ultralytics/cfg`](https://github.com/ultralytics/ultralytics/tree/main/ultralytics/cfg), где находятся все основные конфигурационные файлы. Это ключевое место для начала работы и&nbsp;кастомизации, так как здесь настраиваются параметры моделей, процессы обучения и&nbsp;инференса, а&nbsp;также другие важные аспекты.  

Ниже приведён простой пример callback для лучшего понимания:


In [ ]:
class LoggerCallback:
    def on_train_start(self, state):
        print("Training started!")
        print(f"Initial state: Epoch {state['epoch']}, Accuracy {state['accuracy']:.2f}")

    def on_epoch_start(self, state):
        print(f"Epoch {state['epoch']} started. Current accuracy: {state['accuracy']:.2f}")

    def on_epoch_end(self, state):
        print(f"Epoch {state['epoch']} ende d. Final accuracy: {state['accuracy']:.2f}")

    def on_train_end(self, state):
        print("Training completed!")
        print(f"Final state: Epoch {state['epoch']}, Accuracy {state['accuracy']:.2f}")


class Trainer:
    def __init__(self, epochs, enable_logging=False):
        self.epochs = epochs
        self.callbacks = []
        self.state = {"epoch": 0, "accuracy": 0.0}
        self.enable_logging = enable_logging

        if self.enable_logging:
            self.add_callback(LoggerCallback())

    def add_callback(self, callback):
        self.callbacks.append(callback)

    def train(self):
        for callback in self.callbacks:
            callback.on_train_start(self.state)

        for epoch in range(self.epochs):
            self.state["epoch"] = epoch + 1
            for callback in self.callbacks:
                callback.on_epoch_start(self.state)

            self.state["accuracy"] = 0.8 + 0.02 * epoch
            print(f"Training... Epoch {self.state['epoch']}, Accuracy: {self.state['accuracy']:.2f}")

            for callback in self.callbacks:
                callback.on_epoch_end(self.state)

        for callback in self.callbacks:
            callback.on_train_end(self.state)

In [ ]:
trainer = Trainer(epochs=3, enable_logging=True)
trainer.train()

**Использование архитектуры с обратными вызовами (callbacks):**  

Преимущества:
- гибкость: сallbacks позволяют добавлять и&nbsp;менять функции без&nbsp;изменения основного кода, что упрощает адаптацию под&nbsp;разные задачи;
- чистота кода: логика разделена на независимые модули, что улучшает читаемость и&nbsp;удобство поддержки;
- лёгкая интеграция: удобно подключать через callbacks сторонние логики, такие как логирование, мониторинг и&nbsp;другие.

Недостатки:
- сложная отладка: многоуровневые callbacks могут усложнить отладку, особенно если порядок их выполнения не очевиден;
- риск конфликтов: неправильное использование callbacks может привести к&nbsp;конфликтам и&nbsp;непредсказуемому поведению;
- скрытая логика: чрезмерное использование callbacks усложняет понимание общего процесса, так как часть логики вынесена за пределы основного кода.



Итак, мы узнали, что
- Callbacks, которые испольуются в Ultralytics, позволяют пользователю легко добавлять собственный функционал в работу
- Ultralytics репозиторий построен на конфигах, которые задают тут модели, датасеты и пайплайн обучения

Давайте теперь разбираться с этими конфигами.

<a id='section3'></a>
## **3. Разбираемся с конфигурацией моделей**

Прежде чем погружаться в&nbsp;разбор самого конфига модели, важно внимательно изучить визуализацию архитектуры, предоставленную Ultralytics. Это поможет лучше понять, как устроена модель и&nbsp;какие слои в&nbsp;ней используются. К сожалению, официальной статьи по YOLOv8 нет, но данная визуализация служит отличной отправной точкой для анализа.

Визуализация взята [отсюда](https://github.com/ultralytics/ultralytics/issues/189). 

<img src="assets/yolov8_arch.jpg" alt="YOLOv8 Architecture" width="1000">

Для того чтобы разобраться, как устроена архитектура моделей в&nbsp;Ultralytics, давайте детально изучим конфигурационный файл для составления YOLOv8 модели. Этот файл является базовым шаблоном для построения всех вариаций YOLOv8 моделей и&nbsp;находится по пути [`ultralytics/cfg/models/v8/yolov8.yaml`](https://github.com/ultralytics/ultralytics/blob/main/ultralytics/cfg/models/v8/yolov8.yaml):
```yaml
# Ultralytics YOLO 🚀, AGPL-3.0 license
# YOLOv8 object detection model with P3-P5 outputs. For Usage examples see https://docs.ultralytics.com/tasks/detect

# Parameters
nc: 80 # number of classes
scales: # model compound scaling constants, i.e. 'model=yolov8n.yaml' will call yolov8.yaml with scale 'n'
  # [depth, width, max_channels]
  n: [0.33, 0.25, 1024] # YOLOv8n summary: 225 layers,  3157200 parameters,  3157184 gradients,   8.9 GFLOPs
  s: [0.33, 0.50, 1024] # YOLOv8s summary: 225 layers, 11166560 parameters, 11166544 gradients,  28.8 GFLOPs
  m: [0.67, 0.75, 768] # YOLOv8m summary: 295 layers, 25902640 parameters, 25902624 gradients,  79.3 GFLOPs
  l: [1.00, 1.00, 512] # YOLOv8l summary: 365 layers, 43691520 parameters, 43691504 gradients, 165.7 GFLOPs
  x: [1.00, 1.25, 512] # YOLOv8x summary: 365 layers, 68229648 parameters, 68229632 gradients, 258.5 GFLOPs

# YOLOv8.0n backbone
backbone:
  # [from, repeats, module, args]
  - [-1, 1, Conv, [64, 3, 2]] # 0-P1/2
  - [-1, 1, Conv, [128, 3, 2]] # 1-P2/4
  - [-1, 3, C2f, [128, True]]
  - [-1, 1, Conv, [256, 3, 2]] # 3-P3/8
  - [-1, 6, C2f, [256, True]]
  - [-1, 1, Conv, [512, 3, 2]] # 5-P4/16
  - [-1, 6, C2f, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]] # 7-P5/32
  - [-1, 3, C2f, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]] # 9

# YOLOv8.0n head
head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, Concat, [1]] # cat backbone P4
  - [-1, 3, C2f, [512]] # 12

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 4], 1, Concat, [1]] # cat backbone P3
  - [-1, 3, C2f, [256]] # 15 (P3/8-small)

  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 12], 1, Concat, [1]] # cat head P4
  - [-1, 3, C2f, [512]] # 18 (P4/16-medium)

  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 9], 1, Concat, [1]] # cat head P5
  - [-1, 3, C2f, [1024]] # 21 (P5/32-large)

  - [[15, 18, 21], 1, Detect, [nc]] # Detect(P3, P4, P5)
```

Конфигурационный файл YOLO разделён на три основные части:
1. `Parameters` — общие настройки модели;
2. `Backbone` — часть сети, отвечающая за извлечение признаков;
3. `Head` — часть сети, отвечающая за предсказания.

Далее пойдём по порядку:

**Number of Classes (`nc`):**
- определяет количество классов для&nbsp;детекции (перед началом обучения автоматически подставит нужное количество классов на основе конфига разметки);
- в базовой конфигурации установлено 80 классов (датасет COCO);
- влияет на размер выходного слоя Detect.

**Параметры масштабирования (`scales`)**  
Определяет различные варианты модели через три параметра `[depth_multiple, width_multiple, max_channels]`. Эти параметры позволяют создавать модели разной сложности — от лёгких и&nbsp;быстрых до тяжёлых и&nbsp;точных. Выбор версии модели зависит от ваших вычислительных ресурсов и&nbsp;требований задачи. Подробнее про них:
- `depth_multiple` — множитель глубины сети, который определяет, сколько раз повторять блоки. Например, если в&nbsp;конфиге указано 6 повторений блока, то для&nbsp;nano-версии (0,33): 6 * 0,33 = 2 повторения;
- `width_multiple` — множитель количества каналов. Например, если указано 256 каналов, то для&nbsp;nano-версии (0,25): 256 * 0,25 = 64 канала;
- `max_channels` — ограничение на максимальное количество каналов в&nbsp;сети, который ограничивает рост сети по ширине. В больших моделях по умолчанию используются меньшие значения max_channels, так как они компенсируют это б*о*льшей глубиной сети.

```yaml
scales: # model compound scaling constants, i.e. 'model=yolov8n.yaml' will call yolov8.yaml with scale 'n'
  # [depth, width, max_channels]
  n: [0.33, 0.25, 1024] # YOLOv8n summary: 225 layers,  3157200 parameters,  3157184 gradients,   8.9 GFLOPs
  s: [0.33, 0.50, 1024] # YOLOv8s summary: 225 layers, 11166560 parameters, 11166544 gradients,  28.8 GFLOPs
  m: [0.67, 0.75, 768] # YOLOv8m summary: 295 layers, 25902640 parameters, 25902624 gradients,  79.3 GFLOPs
  l: [1.00, 1.00, 512] # YOLOv8l summary: 365 layers, 43691520 parameters, 43691504 gradients, 165.7 GFLOPs
  x: [1.00, 1.25, 512] # YOLOv8x summary: 365 layers, 68229648 parameters, 68229632 gradients, 258.5 GFLOPs
```
Все значения GFLOPs и&nbsp;количество параметров рассчитаны для&nbsp;входного изображения размером 640x640 пикселей.

**Backbone:**
каждый слой описывается в формате `[from, repeats, module, args]`:
- `from` — указывает, откуда берётся вход для&nbsp;текущего слоя. Можно указать конкретный индекс слоя, а -1 означает с предыдущего слоя;
- `repeats` — сколько раз повторить данный блок. Не забываем, что умножается на&nbsp;depth_multiple из&nbsp;scales;
- `module` — название используемого модуля (например, Conv — обычная свёртка, C2f — блок с&nbsp;cross-connections, ...);
- `args` — аргументы зависят от&nbsp;типа модуля. Например, для&nbsp;[-1, 1, SPPF, [1024, 5]] первый параметр (128) — количество выходных каналов, а&nbsp;второй (True) — использовать ли skip-connection.

**Дополнительно**:  
- `Feature Pyramid (P-слои)` — разные уровни карт признаков в&nbsp;сети, где P1/2 означает уменьшение размера в 2 раза от исходного (320x320 = (640x640)/2); P2/4 — в&nbsp;4 раза и&nbsp;так далее. Маленькие объекты лучше детектируются на ранних P-слоях (P3/8) с высоким разрешением, а большие объекты — на поздних (P5/32) с большим рецептивным полем;
- `C2f блок` — улучшенная версия CSP (Cross Stage Partial) блока, где входной тензор разделяется на две части для&nbsp;оптимизации вычислений. Первая часть проходит через несколько свёрточных слоёв, вторая идёт напрямую, после чего они объединяются;
- `SPPF модуль` — эффективная версия Spatial Pyramid Pooling для&nbsp;извлечения контекстной информации разного масштаба. Последовательно применяет один и&nbsp;тот же maxpool три раза. Использует свёртки 1x1 в&nbsp;начале для&nbsp;уменьшения каналов и&nbsp;в конце для&nbsp;обработки конкатенированных результатов;
- `Upsampling` — операция увеличения пространственного разрешения карт признаков в&nbsp;N раз с помощью интерполяции методом ближайшего соседа. Используется в&nbsp;голове сети для&nbsp;объединения признаков разного масштаба, позволяя комбинировать низкоуровневые детали с высокоуровневым контекстом;
- `Detect слой` — финальный слой сети, который преобразует карты признаков в&nbsp;предсказания объектов. Принимает карты признаков разного масштаба и&nbsp;для каждой ячейки грида предсказывает: координаты бокса в&nbsp;относительном формате (x,y,w,h от&nbsp;0 до&nbsp;1), уверенность обнаружения объекта и&nbsp;вероятности классов.

Особенное внимание хочется уделить **SPPF** блоку (Spatial Pyramid Pooling - Fast), который является оптимизированной версией SPP. Этот блок позволяет эффективно извлекать признаки из входных данных, независимо от их размера, за счёт объединения информации на разных масштабах. SPPF снижает вычислительную нагрузку, улучшает качество признаков и&nbsp;обеспечивает гибкость, поддерживая работу с изображениями различных размеров.

Ниже представлена визуализация оригинального **SPP** из <a href="https://arxiv.org/pdf/1406.4729">статьи</a>:  
<img src="assets/SPP_block.png" alt="SPP block" width="600">


Итак, мы разобрали: 

- как устроена сама модель YOLOv8 и какие у неё есть основные блоки и слои
- как устроен конфиг, задающий архитектуру сети
    
Теперь, когда у нас есть модель, её надо как то учить.

<a id='section5'></a>
## **4. Разбираемся с конфигурацией обучения**

[`default.yaml`](https://github.com/ultralytics/ultralytics/blob/main/ultralytics/cfg/default.yaml) — базовый конфигурационный файл Ultralytics YOLO, который содержит все основные параметры для обучения модели:
- гиперпараметры обучения,
- аугментации данных,
- настройки оптимизатора.

Понимание структуры этого файла критически важно для&nbsp;успешной настройки модели под конкретную задачу:

```yaml
# Ultralytics YOLO 🚀, AGPL-3.0 license
# Default training settings and hyperparameters for medium-augmentation COCO training

task: detect # (str) YOLO task, i.e. detect, segment, classify, pose, obb
mode: train # (str) YOLO mode, i.e. train, val, predict, export, track, benchmark

# Train settings -------------------------------------------------------------------------------------------------------
model: # (str, optional) path to model file, i.e. yolov8n.pt, yolov8n.yaml
data: # (str, optional) path to data file, i.e. coco8.yaml
epochs: 100 # (int) number of epochs to train for
time: # (float, optional) number of hours to train for, overrides epochs if supplied
patience: 100 # (int) epochs to wait for no observable improvement for early stopping of training
batch: 16 # (int) number of images per batch (-1 for AutoBatch)
imgsz: 640 # (int | list) input images size as int for train and val modes, or list[h,w] for predict and export modes
save: True # (bool) save train checkpoints and predict results
save_period: -1 # (int) Save checkpoint every x epochs (disabled if < 1)
cache: False # (bool) True/ram, disk or False. Use cache for data loading
device: # (int | str | list, optional) device to run on, i.e. cuda device=0 or device=0,1,2,3 or device=cpu
workers: 8 # (int) number of worker threads for data loading (per RANK if DDP)
project: # (str, optional) project name
name: # (str, optional) experiment name, results saved to 'project/name' directory
exist_ok: False # (bool) whether to overwrite existing experiment
pretrained: True # (bool | str) whether to use a pretrained model (bool) or a model to load weights from (str)
optimizer: auto # (str) optimizer to use, choices=[SGD, Adam, Adamax, AdamW, NAdam, RAdam, RMSProp, auto]
verbose: True # (bool) whether to print verbose output
seed: 0 # (int) random seed for reproducibility
deterministic: True # (bool) whether to enable deterministic mode
single_cls: False # (bool) train multi-class data as single-class
rect: False # (bool) rectangular training if mode='train' or rectangular validation if mode='val'
cos_lr: False # (bool) use cosine learning rate scheduler
close_mosaic: 10 # (int) disable mosaic augmentation for final epochs (0 to disable)
resume: False # (bool) resume training from last checkpoint
amp: True # (bool) Automatic Mixed Precision (AMP) training, choices=[True, False], True runs AMP check
fraction: 1.0 # (float) dataset fraction to train on (default is 1.0, all images in train set)
profile: False # (bool) profile ONNX and TensorRT speeds during training for loggers
freeze: None # (int | list, optional) freeze first n layers, or freeze list of layer indices during training
multi_scale: False # (bool) Whether to use multiscale during training
# Segmentation
overlap_mask: True # (bool) masks should overlap during training (segment train only)
mask_ratio: 4 # (int) mask downsample ratio (segment train only)
# Classification
dropout: 0.0 # (float) use dropout regularization (classify train only)

# Val/Test settings ----------------------------------------------------------------------------------------------------
val: True # (bool) validate/test during training
split: val # (str) dataset split to use for validation, i.e. 'val', 'test' or 'train'
save_json: False # (bool) save results to JSON file
save_hybrid: False # (bool) save hybrid version of labels (labels + additional predictions)
conf: # (float, optional) object confidence threshold for detection (default 0.25 predict, 0.001 val)
iou: 0.7 # (float) intersection over union (IoU) threshold for NMS
max_det: 300 # (int) maximum number of detections per image
half: False # (bool) use half precision (FP16)
dnn: False # (bool) use OpenCV DNN for ONNX inference
plots: True # (bool) save plots and images during train/val

# Predict settings -----------------------------------------------------------------------------------------------------
source: # (str, optional) source directory for images or videos
vid_stride: 1 # (int) video frame-rate stride
stream_buffer: False # (bool) buffer all streaming frames (True) or return the most recent frame (False)
visualize: False # (bool) visualize model features
augment: False # (bool) apply image augmentation to prediction sources
agnostic_nms: False # (bool) class-agnostic NMS
classes: # (int | list[int], optional) filter results by class, i.e. classes=0, or classes=[0,2,3]
retina_masks: False # (bool) use high-resolution segmentation masks
embed: # (list[int], optional) return feature vectors/embeddings from given layers

# Visualize settings ---------------------------------------------------------------------------------------------------
show: False # (bool) show predicted images and videos if environment allows
save_frames: False # (bool) save predicted individual video frames
save_txt: False # (bool) save results as .txt file
save_conf: False # (bool) save results with confidence scores
save_crop: False # (bool) save cropped images with results
show_labels: True # (bool) show prediction labels, i.e. 'person'
show_conf: True # (bool) show prediction confidence, i.e. '0.99'
show_boxes: True # (bool) show prediction boxes
line_width: # (int, optional) line width of the bounding boxes. Scaled to image size if None.

# Export settings ------------------------------------------------------------------------------------------------------
format: torchscript # (str) format to export to, choices at https://docs.ultralytics.com/modes/export/#export-formats
keras: False # (bool) use Kera=s
optimize: False # (bool) TorchScript: optimize for mobile
int8: False # (bool) CoreML/TF INT8 quantization
dynamic: False # (bool) ONNX/TF/TensorRT: dynamic axes
simplify: True # (bool) ONNX: simplify model using `onnxslim`
opset: # (int, optional) ONNX: opset version
workspace: 4 # (int) TensorRT: workspace size (GB)
nms: False # (bool) CoreML: add NMS

# Hyperparameters ------------------------------------------------------------------------------------------------------
lr0: 0.01 # (float) initial learning rate (i.e. SGD=1E-2, Adam=1E-3)
lrf: 0.01 # (float) final learning rate (lr0 * lrf)
momentum: 0.937 # (float) SGD momentum/Adam beta1
weight_decay: 0.0005 # (float) optimizer weight decay 5e-4
warmup_epochs: 3.0 # (float) warmup epochs (fractions ok)
warmup_momentum: 0.8 # (float) warmup initial momentum
warmup_bias_lr: 0.1 # (float) warmup initial bias lr
box: 7.5 # (float) box loss gain
cls: 0.5 # (float) cls loss gain (scale with pixels)
dfl: 1.5 # (float) dfl loss gain
pose: 12.0 # (float) pose loss gain
kobj: 1.0 # (float) keypoint obj loss gain
label_smoothing: 0.0 # (float) label smoothing (fraction)
nbs: 64 # (int) nominal batch size
hsv_h: 0.015 # (float) image HSV-Hue augmentation (fraction)
hsv_s: 0.7 # (float) image HSV-Saturation augmentation (fraction)
hsv_v: 0.4 # (float) image HSV-Value augmentation (fraction)
degrees: 0.0 # (float) image rotation (+/- deg)
translate: 0.1 # (float) image translation (+/- fraction)
scale: 0.5 # (float) image scale (+/- gain)
shear: 0.0 # (float) image shear (+/- deg)
perspective: 0.0 # (float) image perspective (+/- fraction), range 0-0.001
flipud: 0.0 # (float) image flip up-down (probability)
fliplr: 0.5 # (float) image flip left-right (probability)
bgr: 0.0 # (float) image channel BGR (probability)
mosaic: 1.0 # (float) image mosaic (probability)
mixup: 0.0 # (float) image mixup (probability)
copy_paste: 0.0 # (float) segment copy-paste (probability)
copy_paste_mode: "flip" # (str) the method to do copy_paste augmentation (flip, mixup)
auto_augment: randaugment # (str) auto augmentation policy for classification (randaugment, autoaugment, augmix)
erasing: 0.4 # (float) probability of random erasing during classification training (0-0.9), 0 means no erasing, must be less than 1.0.
crop_fraction: 1.0 # (float) image crop fraction for classification (0.1-1), 1.0 means no crop, must be greater than 0.

# Custom config.yaml ---------------------------------------------------------------------------------------------------
cfg: # (str, optional) for overriding defaults.yaml

# Tracker settings ------------------------------------------------------------------------------------------------------
tracker: botsort.yaml # (str) tracker type, choices=[botsort.yaml, bytetrack.yaml]
```

Итак, в конфиге обучения задаются все основные параметры обучения/валидации/тестирования/визуализации модели. Это основное наше место работы с YOLO детектором. 

Давайте теперь разберем некоторые механизмы, которые участвуют в процессе обучения.

<a id='section5'></a>
## **5. Полезные механизмы для обучения**

<a id='section5-1'></a>
### **5.1. Scale и Multi_scale**

На первый взгляд параметры `scale` и `multi_scale` в&nbsp;YOLO кажутся похожими — оба меняют размер изображений во&nbsp;время тренировки. Однако механизмы их работы и&nbsp;конечный эффект существенно различаются!

- **`scale` augmentation** (`scale: 0.5`) — объект может стать больше или&nbsp;меньше относительно кадра;
- **`multi-scale` training** (`multi_scale: True`) — всё изображение целиком будет больше или меньше.

Это два разных подхода к&nbsp;обучению модели работы с&nbsp;размерами:

- **`scale`** учит модель различать объекты разного масштаба в&nbsp;контексте одной сцены;
- **`multi_scale`** помогает модели адаптироваться к&nbsp;изображениям разного разрешения.


Масштабирование с использованием параметра `scale` случайным образом изменяет размер изображения в&nbsp;диапазоне [1 - scale, 1 + scale]. При **scale = 0.5**:  
- минимальный масштаб: **0.5х** от&nbsp;оригинального размера;  
- максимальный масштаб: **1.5х** от&nbsp;оригинального размера.  

Каждое изображение масштабируется независимо. Например, для&nbsp;оригинального изображения размером 640x640 случайное масштабирование может уменьшить его до&nbsp;320x320, увеличить до 960x960 или быть где-то в&nbsp;этом диапазоне (рандомно). После масштабирования изображение ресайзится обратно к&nbsp;размеру батча (например, 640x640). В&nbsp;результате объекты на&nbsp;изображении становятся визуально больше или&nbsp;меньше относительно друг друга.

Масштабирование с использованием **`multi_scale`** используется для&nbsp;обучения модели на&nbsp;изображениях разных размеров, чтобы улучшить её способность обрабатывать входы с&nbsp;различным разрешением. Это означает, что на&nbsp;каждом шаге тренировки размер входного изображения случайным образом выбирается из&nbsp;предопределённого диапазона.

В случае примера ниже этот трюк работает благодаря тому, что в&nbsp;сети используется адаптивный пулинг (AdaptiveAvgPool), который всегда приводит карты признаков к&nbsp;нужному фиксированному размеру независимо от&nbsp;размера входного изображения. Таким образом, выходной размер модели остаётся постоянным:

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(SimpleCNN, self).__init__()
        self.conv = nn.Conv2d(3, 16, 3, padding=1)
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(16, num_classes)

    def forward(self, x):
        x = F.relu(self.conv(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

# Создаём модель на 10 классов
model = SimpleCNN(num_classes=10)
model.eval()

# Создадим игрушечное изображение
img = Image.new('RGB', (800, 800), color='white')

# Размеры, которые мы будем перебирать
sizes = [224, 320, 416, 512, 608]

for size in sizes:
    transform = Resize((size, size))
    img_resized = transform(img)
    # Переводим в нужный формат перед подачей в сеть: [1, 3, H, W]
    img_tensor = ToTensor()(img_resized).unsqueeze(0)

    with torch.no_grad():
        output = model(img_tensor)

    print(f'Входной размер: {size}x{size}, Выходной размер: {output.shape}')


В случае архитектуры семейства YOLO при&nbsp;изменении размера изображения количество прогнозируемых боксов будет изменяться. Это происходит потому, что YOLO использует свёрточные слои для&nbsp;обработки изображений, и итоговый размер выходной карты признаков зависит от&nbsp;размера входа. Больше разрешение — больше сетка предсказаний, следовательно, больше боксов:

In [ ]:
class SimpleCNNLikeYOLO(nn.Module):
    def __init__(self):
        super(SimpleCNNLikeYOLO, self).__init__()
        # Базовый конволюционный слой
        self.conv = nn.Conv2d(3, 16, 3, stride=2, padding=1)

        # Слой детекции (упрощённый)
        # Для каждой ячейки сетки предсказываем:
        # - 4 координаты бокса (x, y, w, h)
        # - 1 значение уверенности (objectness)
        # - 10 классов
        # Итого: 15 значений на каждый бокс
        self.detect = nn.Conv2d(16, 15, 1)

    def forward(self, x):
        x = F.relu(self.conv(x))
        # Детекции на выходе будут иметь размер [batch, 15, H/stride, W/stride]
        x = self.detect(x)
        return x

# Создаём модель
model = SimpleCNNLikeYOLO()
model.eval()

# Создадим игрушечное изображение
img = Image.new('RGB', (800, 800), color='white')

# Размеры, которые мы будем перебирать
sizes = [224, 320, 416, 512, 608]

for size in sizes:
    transform = Resize((size, size))
    img_resized = transform(img)
    # Переводим в нужный формат перед подачей в сеть (1, 3, H, W)
    img_tensor = ToTensor()(img_resized).unsqueeze(0)

    with torch.no_grad():
        output = model(img_tensor)

    # Считаем количество предсказанных боксов
    grid_size = output.shape[2]  # H/stride
    num_boxes = grid_size * grid_size

    print(f'Размер входа: {size}x{size}')
    print(f'Размер выхода: {output.shape}')
    print(f'Размер сетки: {grid_size}x{grid_size}')
    print(f'Всего предсказаний: {num_boxes} боксов')
    print('-' * 40)

Итог:
- `scale` фокусируется на улучшении способности модели различать объекты разных размеров в&nbsp;одной сцене, случайно масштабируя объекты относительно фона;
- `multi_scale` улучшает способность модели работать с&nbsp;изображениями разных разрешений, обучая её на изображениях с&nbsp;различным размером, сохраняя пропорции между объектами.  

То есть `scale` оптимизирует работу с&nbsp;объектами разных масштабов, а&nbsp;`multi_scale` — с&nbsp;изображениями разных разрешений.

<a id='section5-2'></a>
### **5.2. Механизмы mosaic и close_mosaic**

`mosaic` — это метод агрегации четырёх случайных изображений в&nbsp;одно, впервые представленный в&nbsp;YOLOv4. Этот подход значительно увеличивает разнообразие данных и помогает модели лучше адаптироваться к&nbsp;различным условиям. Из четырёх изображений формируется одно «мозаичное» изображение, где каждый фрагмент соответствует части одного из&nbsp;исходных изображений. Данная аугментация увеличивает плотность объектов на&nbsp;изображении и&nbsp;помогает обучить модель на&nbsp;данных с&nbsp;разными пропорциями и&nbsp;контекстами, улучшая её способность к генерализации.

`close_mosaic` — это не отдельный вариант техники mosaic, а, скорее, практика отключения на финальных этапах обучения модели. Суть в&nbsp;том, что в&nbsp;начале и&nbsp;середине обучения используется обычный mosaic для&nbsp;лучшей генерализации, а ближе к&nbsp;концу обучения mosaic постепенно отключается, чтобы модель лучше адаптировалась к&nbsp;реальным условиям использования, где изображения обрабатываются целиком, без&nbsp;мозаики. Это помогает найти баланс между преимуществами mosaic для&nbsp;общего обучения и&nbsp;генерализации и&nbsp;необходимостью финальной настройки на&nbsp;целевой формат работы без&nbsp;такой сложной аугментации.


In [ ]:
# Используем следующие пути изображений и их относительные координаты объектов (в формате YOLO):
image_paths = [
    '/kaggle/input/datasets/andreykurdyubov/cv-lesson7-assets/assets/1.jpg',
    '/kaggle/input/datasets/andreykurdyubov/cv-lesson7-assets/assets/2.jpg',
    '/kaggle/input/datasets/andreykurdyubov/cv-lesson7-assets/assets/3.jpg',
    '/kaggle/input/datasets/andreykurdyubov/cv-lesson7-assets/assets/4.jpg'
]

boxes_rel = [
    [0.5697115384615384, 0.7271634615384616, 0.5492788461538461, 0.5432692307692307],
    [0.36177884615384615, 0.71875, 0.5168269230769231, 0.22355769230769232],
    [0.18870192307692307, 0.6177884615384616, 0.3389423076923077, 0.2764423076923077],
    [0.3557692307692308, 0.6995192307692307, 0.22355769230769232, 0.09254807692307693]
]

In [ ]:
def load_image_and_box(img_path, box):
    """
    Загрузка изображения и конвертация относительных координат бокса в абсолютные
    Args:
        img_path: путь к изображению
        box: [x_center, y_center, width, height] в относительных координатах [0-1] (YOLO формат)
    Returns:
        img: изображение в RGB
        box: [x1, y1, x2, y2] в абсолютных координатах
    """
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    x, y, width, height = box

    # Конвертация из относительных в абсолютные координаты
    x1 = int((x - width/2) * w)
    y1 = int((y - height/2) * h)
    x2 = int((x + width/2) * w)
    y2 = int((y + height/2) * h)
    return img, (x1, y1, x2, y2)

def mosaic_ultralytics(images, boxes, target_size=640):
    """
    Простая имплементация аугментации мозаики из 4 изображений, как в ultralytics

    Args:
        images: список из 4 изображений
        boxes: список из 4 боксов в формате [x1, y1, x2, y2] (абсолютные координаты)
        target_size: размер выходного изображения (по умолчанию 640)
    Returns:
        result_img: мозаика размером (target_size x target_size)
        new_boxes: список трансформированных боксов
    """

    result_img = np.full((target_size, target_size, 3), 114, dtype=np.uint8)
    new_boxes = []

    cx = int(random.uniform(target_size//2, 2*target_size//3))
    cy = int(random.uniform(target_size//2, 2*target_size//3))

    for i, (img, box) in enumerate(zip(images, boxes)):
        if i == 0:
            x1a, y1a, x2a, y2a = 0, 0, cx, cy
        elif i == 1:
            x1a, y1a, x2a, y2a = cx, 0, target_size, cy
        elif i == 2:
            x1a, y1a, x2a, y2a = 0, cy, cx, target_size
        elif i == 3:
            x1a, y1a, x2a, y2a = cx, cy, target_size, target_size

        h, w = img.shape[:2]
        scale = min((x2a - x1a) / w, (y2a - y1a) / h)

        new_w = int(w * scale)
        new_h = int(h * scale)
        resized_img = cv2.resize(img, (new_w, new_h))

        new_box = [
                x1a + (box[0] / w) * (x2a - x1a),
                y1a + (box[1] / h) * (y2a - y1a),
                x1a + (box[2] / w) * (x2a - x1a),
                y1a + (box[3] / h) * (y2a - y1a)
        ]
        new_boxes.append(new_box)

        result_img[y1a:y2a, x1a:x2a] = cv2.resize(resized_img, (x2a - x1a, y2a - y1a))

    return result_img, new_boxes

In [ ]:
images_and_boxes = [load_image_and_box(path, box) for path, box in zip(image_paths, boxes_rel)]
images = [item[0] for item in images_and_boxes]
boxes = [item[1] for item in images_and_boxes]

colors = list(mcolors.TABLEAU_COLORS.values())
box_colors = [colors[0], colors[0], colors[1], colors[2]]

fig, axs = plt.subplots(1, 4, figsize=(20, 5))

for i, (img, box) in enumerate(zip(images, boxes)):
    axs[i].imshow(img)
    axs[i].plot([box[0], box[2], box[2], box[0], box[0]],
                [box[1], box[1], box[3], box[3], box[1]],
                color=box_colors[i], linewidth=2)
    axs[i].set_title(f"Image {i+1}", fontsize=14)
    axs[i].axis('off')

In [ ]:
images_and_boxes = [load_image_and_box(path, box) for path, box in zip(image_paths, boxes_rel)]
images = [item[0] for item in images_and_boxes]
boxes = [item[1] for item in images_and_boxes]

mosaic_img, mosaic_boxes = mosaic_ultralytics(images, boxes)

# Цвета для боксов
colors = list(mcolors.TABLEAU_COLORS.values())
box_colors = [colors[0], colors[0], colors[1], colors[2]]

plt.figure(figsize=(12, 12))
plt.imshow(mosaic_img)
for box, color in zip(mosaic_boxes, box_colors):
    plt.plot([box[0], box[2], box[2], box[0], box[0]],
             [box[1], box[1], box[3], box[3], box[1]], color=color, linewidth=2)

    plt.title(f"Mosaic image (1-4)", fontsize=14)
plt.axis('off')
plt.show()

<a id='section5-3'></a>
### **5.3. Распределение весов у&nbsp;лоссов во&nbsp;время обучения**

Глобально YOLO loss состоит из трёх частей, каждая из&nbsp;которых отвечает за&nbsp;свою задачу:

- `Class Loss` — помогает модели правильно классифицировать объекты, минимизируя разницу между предсказанными и истинными метками классов, это просто Cross-Entropy.
- `Box Loss` — отвечает за точное позиционирование и размер bounding boxes, это CIoU Loss - Complete IoU Loss;
- `Distribution Focal Loss` — улучшает локализацию объектов, помогая модели точнее определять границы боксов.

<img src="assets/loss.png" alt="loss" width="600">

Почитать локализационные части лосса можно тут: [Box Loss](https://learnopencv.com/iou-loss-functions-object-detection/), [Distributed Focal Loss](https://github.com/ultralytics/ultralytics/issues/6596)

Используя весовые коэффициенты, мы можем контролировать вклад каждой компоненты лосса в&nbsp;общий лосс.  

Это позволяет:  
- **балансировать обучение** — увеличивая или уменьшая веса, можно заставить модель больше фокусироваться на&nbsp;определённых аспектах задачи (например, точности локализации объектов или правильной классификации);
- **улучшать сходимость** — правильная настройка весов способствует более стабильной и быстрой сходимости модели во время обучения;
- **адаптироваться к задаче** — в зависимости от специфики данных и&nbsp;целей обучения можно настраивать веса для&nbsp;достижения наилучших результатов.

В задачах машинного обучения часто используется несколько функций потерь. В&nbsp;нашем случае их три, и их всегда можно комбинировать с&nbsp;различными коэффициентами, чтобы получить оптимальный результат для&nbsp;конкретной задачи.  Не бойтесь экспериментировать с коэффициентами — это важная часть настройки модели. Подбор правильных весов для&nbsp;каждой функции потерь может значительно улучшить качество обучения!


<a id='section5-4'></a>
### **5.4. Критерий качества модели**

*Расположение fitness (критерий оценки) функции*  
Основная fitness-функция находится в&nbsp;файле [`ultralytics/utils/metrics.py`](https://github.com/ultralytics/ultralytics/blob/main/ultralytics/utils/metrics.py). Она используется для оценки качества модели во время валидации и сохраняет модель в зависимости от того, что нам важнее:

```python
def fitness(self):
    """Model fitness as a weighted combination of metrics."""
    w = [0.0, 0.0, 0.1, 0.9]  # weights for [P, R, mAP@0.5, mAP@0.5:0.95]
    return (np.array(self.mean_results()) * w).sum()
```

**Компоненты метрики:**  
`P (Precision)` — точность (вес 0.0);   
    
`R (Recall)` — полнота (вес 0.0);
    
`mAP@0.5` — mean Average Precision при IoU=0.5 (вес 0.1);
    
`mAP@0.5:0.95` — mean Average Precision при IoU от 0.5 до 0.95 (вес 0.9).

В Ultralytics есть также механизм ранней остановки (early stopping), который прекращает обучение, если fitness-метрика не улучшается в течение заданного количества эпох. Кроме того, если метрика длительное время не&nbsp;улучшается, происходит уменьшение learning rate (если установлен соответствующий scheduler).

Итак, мы обсудили важные параметры обучения:
- механизмы `scale` и `multi_scale`
- `mosaic` аугментацию
- сложносоставной лосс YOLO и балансирование его составляющих
- критерии качества для выбора лучшей модели

<a id='section6'></a>
## **6. Заводим обучение YOLO**

Обычно в DL обучение нейронных сетей всё же не происходит в ноутбуках, поэтому и в YOLO наиболее удобные методы используют командную строку. Инструкцию можно найти [здесь](https://docs.ultralytics.com/ru/quickstart/#use-ultralytics-with-cli). 

Мы же сейчас попробуем простейшее обучение в `Python` запустить. 

In [ ]:
from ultralytics import YOLO

In [ ]:
# Создаем YOLO модель from scratch, конфиг: https://github.com/ultralytics/ultralytics/blob/main/ultralytics/cfg/models/v8/yolov8.yaml
model = YOLO('yolov8n.yaml')

In [ ]:
# смотрим на модель
model

In [ ]:
# Подгружаем для неё веса COCO
model = YOLO('yolov8n.pt')

In [ ]:
# Проводим обучение на нано-COCO на 5 эпох, конфиг https://github.com/ultralytics/ultralytics/blob/main/ultralytics/cfg/datasets/coco8.yaml
# Обучение при этом дефолтное по конфигу https://github.com/ultralytics/ultralytics/blob/main/ultralytics/cfg/default.yaml
results = model.train(data='coco8.yaml', epochs=5)

In [ ]:
# Оценим результаты модели на валидации
results = model.val()

In [ ]:
# Посмотрим результат работы на картинке
im = cv2.imread('/kaggle/input/datasets/andreykurdyubov/cv-lesson7-assets/assets/bus.jpg')
results = model.predict(source=im, save=True)

In [ ]:
results[0]

In [ ]:
res = cv2.imread('/kaggle/working/runs/detect/predict/zidane.jpg')
plt.imshow(res[:, :, ::-1])
plt.show()

Класс, модель действительно детектит некоторые объекты (тех классов, что есть в COCO).

Итак, мы поняли
- как инициализировать модель и подгрузить веса
- как завести обучение с дефолтным конфигом 
- как много информации логирует YOLO при обучении
- как провести валидацию модели и отрисовать результаты на отдельной картинке

<a id='section7'></a>
## **7. Знакомимся с ONNX форматом**

In [ ]:
# установим onnx если его нет
%pip install onnx onnxslim

Давайте теперь представим, что нас полностью устраивает качество сетки и мы бы хотели её сохранить в `ONNX` - универсальном формате, который можно использовать для инференеса сети и для дальнейшей её конвертации под конкретные устройства. 

В YOLO это делается очень просто:

In [ ]:
# Экспортируем модель в ONNX 
success = model.export(format='onnx', simplify=True)

Теперь, чтобы использовать `onnx`-модель, нам понадобится `onnxruntime`, его [дока](https://onnxruntime.ai/docs/).

In [ ]:
import onnxruntime as ort

In [ ]:
# Пример можно посмотреть тут: https://github.com/ultralytics/ultralytics/blob/main/examples/YOLOv8-ONNXRuntime/main.py

def preprocess(path):
    img = cv2.imread(path)
    h, w = img.shape[:2]
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    image_data = cv2.resize(img, (640, 640))
    image_data = np.array(image_data) / 255.0
    image_data = np.transpose(image_data, (2, 0, 1))
    image_data = np.expand_dims(image_data, axis=0).astype(np.float32)
    return img, image_data, (w, h)
    
    
# готовим инпут
img, inp, (w, h) = preprocess('/kaggle/input/datasets/andreykurdyubov/cv-lesson7-assets/assets/bus.jpg')

In [ ]:
# создаем сессию
session = ort.InferenceSession('/kaggle/working/runs/detect/train/weights/best.onnx')
model_inputs = session.get_inputs()

# получаем результат работы
outputs = session.run(None, {model_inputs[0].name: inp})

In [ ]:
# посмотрим что будет на выходе
outputs[0]

Это сырые выходы модели, знакомьтесь. Их надо обработать, чтобы получить адекватные боксы

In [ ]:
# источник https://github.com/ultralytics/ultralytics/blob/main/examples/YOLOv8-ONNXRuntime/main.py

def draw_detections(img, box, score, class_id):
    x1, y1, w, h = box
    color_palette = np.random.uniform(0, 255, size=(100, 3))
    color = color_palette[class_id]
    cv2.rectangle(img, (int(x1), int(y1)), (int(x1 + w), int(y1 + h)), color, 2)
    label = f"{class_id}: {score:.2f}"
    (label_width, label_height), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
    label_x = x1
    label_y = y1 - 10 if y1 - 10 > label_height else y1 + 10
    cv2.rectangle(
        img, (label_x, label_y - label_height), (label_x + label_width, label_y + label_height), color, cv2.FILLED
    )
    cv2.putText(img, label, (label_x, label_y), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
        
        
def postprocess(img, img_shape, output):
    outputs = np.transpose(np.squeeze(output[0]))
    rows = outputs.shape[0]

    # завосим списки для боксов, скоров, id
    boxes = []
    scores = []
    class_ids = []

    # коэффициенты для обратного ресайза боксов
    x_factor = img_shape[1] / 640
    y_factor = img_shape[0] / 640

    for i in range(rows):
        # берем скоры классов и выбираем максимальный
        classes_scores = outputs[i][4:]
        max_score = np.amax(classes_scores)

        # сравниваем его с порогом
        if max_score >= 0.5:
            class_id = np.argmax(classes_scores)

            # вытаскиваем bbox
            x, y, w, h = outputs[i][0], outputs[i][1], outputs[i][2], outputs[i][3]

            # пересчитываем координаты
            left = int((x - w / 2) * x_factor)
            top = int((y - h / 2) * y_factor)
            width = int(w * x_factor)
            height = int(h * y_factor)

            # дополняем список
            class_ids.append(class_id)
            scores.append(max_score)
            boxes.append([left, top, width, height])

    # используем NMS, в indices будут id выживших боксов
    indices = cv2.dnn.NMSBoxes(boxes, scores, 0.5, 0.5)
    for i in indices:
        box = boxes[i]
        score = scores[i]
        class_id = class_ids[i]
        draw_detections(img, box, score, class_id)

    return img

In [ ]:
onnx_result = postprocess(img, (h, w), outputs[0])

In [ ]:
plt.imshow(onnx_result)

Та-дам! В результате получили те же боксы, а значит наша `ONNX` модель и правда работает. 

Итак, теперь мы
- можем сконвертировать детектор `YOLO` в `ONNX` формат
- а так же запустить onnx для инференса с помощью `onnxruntime`

Давайте же подведем итоги всей практической части. 

## **Итоги**

В этой практике мы:

- разобрали, как устроены разметки для детекции, 
- изучили структуру проекта Ultralytics и конфигурации моделей, а также рассмотрели важные механизмы обучения. 
- уделили внимание тому, как работают лоссы и как у них распределяются веса. 
- применили все эти знания на практике и запустили небольшое обучение YOLOv8n
- а также сконвертировали модель в ONNX формат и протестировали её с помощью onnxruntime


Ultralytics — мощный инструмент, который позволяет гибко адаптировать модели под ваши задачи. Не бойтесь экспериментировать с настройками и тестировать разные конфигурации. Это не только улучшит ваши модели, но и укрепит ваши навыки как инженера!

Ну а мы сегодня были большими молодцами и закончили разбирать сложнейший блок детекции объектов, 
отточить ваши знания вы сможете в ДЗ, а я на этом прощаюсь. 

See you!

